In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/home-credit-default-risk/sample_submission.csv
/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv
/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv
/kaggle/input/competitions/home-credit-default-risk/application_train.csv
/kaggle/input/competitions/home-credit-default-risk/HomeCredit_columns_description.csv
/kaggle/input/competitions/home-credit-default-risk/application_test.csv
/kaggle/input/competitions/home-credit-default-risk/previous_application.csv
/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv
/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv
/kaggle/input/competitions/home-credit-default-risk/bureau.csv


In [2]:
# Data handling
import pandas as pd
import numpy as np

# Visuals
import matplotlib.pyplot as plt
import seaborn as sns

# ML tools
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report

# Feature selection
from sklearn.feature_selection import SelectFromModel

# XGBoost
from xgboost import XGBClassifier

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Main application data
train = pd.read_csv(
    "/kaggle/input/competitions/home-credit-default-risk/application_train.csv"
)

# Bureau credit history
bureau = pd.read_csv(
    "/kaggle/input/competitions/home-credit-default-risk/bureau.csv"
)

# Bureau monthly history
bureau_balance = pd.read_csv(
    "/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv"
)

# Previous applications
previous = pd.read_csv(
    "/kaggle/input/competitions/home-credit-default-risk/previous_application.csv"
)

# Installment payments
installments = pd.read_csv(
    "/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv"
)

# Credit card balances
credit_card = pd.read_csv(
    "/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv"
)

# POS cash balance
pos_cash = pd.read_csv(
    "/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv"
)

# Check shapes
print(train.shape)
print(bureau.shape)
print(previous.shape)

(307511, 122)
(1716428, 17)
(1670214, 37)


In [4]:
# Numeric bureau columns
bureau_num = bureau.select_dtypes(
    include=["int64", "float64"]
)

# Aggregate bureau data
bureau_agg = bureau_num.groupby(
    "SK_ID_CURR"
).agg([

    "mean",
    "max",
    "min",
    "sum",
    "std"

])

# Rename columns
bureau_agg.columns = [

    "BUREAU_" + "_".join(col).upper()

    for col in bureau_agg.columns
]
# Reset index
bureau_agg = bureau_agg.reset_index()

In [5]:
# Numeric previous application columns
previous_num = previous.select_dtypes(
    include=["int64", "float64"]
)

# Aggregate
previous_agg = previous_num.groupby(
    "SK_ID_CURR"
).agg([

    "mean",
    "max",
    "min",
    "sum",
    "std"

])

# Rename columns
previous_agg.columns = [

    "PREV_" + "_".join(col).upper()

    for col in previous_agg.columns
]

# Reset index
previous_agg = previous_agg.reset_index()

In [6]:
# Payment delay
installments["PAYMENT_DELAY"] = (

    installments["DAYS_ENTRY_PAYMENT"] -
    installments["DAYS_INSTALMENT"]

)

# Numeric installment columns
install_num = installments.select_dtypes(
    include=["int64", "float64"]
)

# Aggregate installments
install_agg = install_num.groupby(
    "SK_ID_CURR"
).agg([

    "mean",
    "max",
    "min",
    "sum",
    "std"

])

# Rename columns
install_agg.columns = [

    "INSTALL_" + "_".join(col).upper()

    for col in install_agg.columns
]

# Reset index
install_agg = install_agg.reset_index()

# Create payment delay
installments["PAYMENT_DELAY"] = (
    installments["DAYS_ENTRY_PAYMENT"] -
    installments["DAYS_INSTALMENT"]
)

# Create late payment flag
installments["LATE_PAYMENT"] = (
    installments["PAYMENT_DELAY"] > 0
).astype(int)

# Create payment difference
installments["PAYMENT_DIFF"] = (
    installments["AMT_INSTALMENT"] -
    installments["AMT_PAYMENT"]
)

# Create payment ratio
installments["PAYMENT_RATIO"] = (
    installments["AMT_PAYMENT"] /
    installments["AMT_INSTALMENT"]
)

# Replace infinity values
installments = installments.replace(
    [np.inf, -np.inf],
    np.nan
)

# Aggregate repayment behavior
repayment_agg = installments.groupby(
    "SK_ID_CURR"
).agg({

    "PAYMENT_DELAY": ["mean", "max", "sum"],
    "LATE_PAYMENT": ["mean", "sum"],
    "PAYMENT_DIFF": ["mean", "max", "min"],
    "PAYMENT_RATIO": ["mean", "min", "max"]

})

# Rename columns
repayment_agg.columns = [
    "REPAY_" + "_".join(col).upper()
    for col in repayment_agg.columns
]

# Reset index
repayment_agg = repayment_agg.reset_index()

In [7]:
# Numeric credit card columns
credit_num = credit_card.select_dtypes(
    include=["int64", "float64"]
)

# Aggregate
credit_agg = credit_num.groupby(
    "SK_ID_CURR"
).agg([

    "mean",
    "max",
    "min",
    "sum",
    "std"

])

# Rename columns
credit_agg.columns = [

    "CC_" + "_".join(col).upper()

    for col in credit_agg.columns
]

# Reset index
credit_agg = credit_agg.reset_index()

In [8]:
# Merge bureau
train = train.merge(
    bureau_agg,
    on="SK_ID_CURR",
    how="left"
)

# Merge previous applications
train = train.merge(
    previous_agg,
    on="SK_ID_CURR",
    how="left"
)

# Merge installments
train = train.merge(
    install_agg,
    on="SK_ID_CURR",
    how="left"
)

# Merge credit card
train = train.merge(
    credit_agg,
    on="SK_ID_CURR",
    how="left"
)

# Merge repayment features
train = train.merge(
    repayment_agg,
    on="SK_ID_CURR",
    how="left"
)

# Final shape
print(train.shape)

(307511, 443)


In [9]:
# Credit to income ratio
train["CREDIT_INCOME_RATIO"] = (
    train["AMT_CREDIT"] /
    train["AMT_INCOME_TOTAL"]
)

# Annuity to income ratio
train["ANNUITY_INCOME_RATIO"] = (
    train["AMT_ANNUITY"] /
    train["AMT_INCOME_TOTAL"]
)

# Credit to goods ratio
train["GOODS_CREDIT_RATIO"] = (
    train["AMT_GOODS_PRICE"] /
    train["AMT_CREDIT"]
)

# Employment age ratio
train["EMPLOYMENT_AGE_RATIO"] = (
    train["DAYS_EMPLOYED"] /
    train["DAYS_BIRTH"]
)

In [10]:
# Target
y = train["TARGET"]

# Remove target and ID
X = train.drop(
    columns=["TARGET", "SK_ID_CURR"]
)

# One hot encoding
X = pd.get_dummies(
    X,
    drop_first=True
)

# Replace infinite values
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

# Fill missing values
X = X.fillna(
    X.median()
)

In [11]:
# Initial model
selector_model = XGBClassifier(

    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    eval_metric="auc",
    n_jobs=-1
)

In [12]:
# Train selector model
selector_model.fit(
    X,
    y
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=-1, num_parallel_tree=None, ...)

In [13]:
# Feature selector
selector = SelectFromModel(

    selector_model,

    threshold="median",

    prefit=True
)

# Transform dataset
X_selected = selector.transform(X)

# Selected feature names
selected_features = X.columns[
    selector.get_support()
]

# Show selected count
print(len(selected_features))

277


In [14]:
# Convert selected data back to dataframe
X_selected = pd.DataFrame(
    X_selected,
    columns=selected_features
)

# Split selected data
X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
# Calculate class imbalance
scale_pos_weight = (
    y_train.value_counts()[0] /
    y_train.value_counts()[1]
)

# Show weight
print(scale_pos_weight)

11.38710976837865


In [16]:
# Model for tuning
xgb = XGBClassifier(
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
    scale_pos_weight=scale_pos_weight
)

In [17]:
# Parameter search space
param_grid = {
    "n_estimators": [500, 800, 1200],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "min_child_weight": [3, 5, 7],
    "gamma": [0, 0.1, 0.3],
    "reg_alpha": [0, 0.5, 1],
    "reg_lambda": [1, 2, 5]
}

In [18]:
# Cross validation setup
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [19]:
# Final optimized XGBoost model
final_model = XGBClassifier(

    # Number of trees
    n_estimators=500,

    # Learning speed
    learning_rate=0.03,

    # Tree depth
    max_depth=4,

    # Prevent overfitting
    min_child_weight=5,

    # Random row sampling
    subsample=0.8,

    # Random feature sampling
    colsample_bytree=0.8,

    # Split control
    gamma=0.1,

    # Regularization
    reg_alpha=0.5,
    reg_lambda=2,

    # Handle imbalance
    scale_pos_weight=scale_pos_weight,

    # Evaluation metric
    eval_metric="auc",

    # Faster training
    tree_method="hist",

    # Use all CPU cores
    n_jobs=-1,

    # Reproducibility
    random_state=42
)

# Train model
final_model.fit(
    X_train,
    y_train
)



XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=0.1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)

In [20]:
# Predict probabilities
probs = final_model.predict_proba(X_test)[:, 1]

# Convert probabilities into predictions
preds = (probs >= 0.5).astype(int)

# Classification report
print(
    classification_report(
        y_test,
        preds
    )
)

# ROC AUC score
auc = roc_auc_score(
    y_test,
    probs
)

print("ROC AUC:", auc)

              precision    recall  f1-score   support

           0       0.96      0.72      0.83     56538
           1       0.18      0.70      0.29      4965

    accuracy                           0.72     61503
   macro avg       0.57      0.71      0.56     61503
weighted avg       0.90      0.72      0.78     61503

ROC AUC: 0.7814832537657834


In [21]:
# Import metric
from sklearn.metrics import precision_recall_curve

# Precision recall values
precision, recall, thresholds = precision_recall_curve(
    y_test,
    probs
)

# Calculate F1 scores
f1_scores = (
    2 * (precision * recall)
) / (precision + recall)

# Best threshold
best_threshold = thresholds[
    f1_scores[:-1].argmax()
]

print("Best threshold:", best_threshold)

Best threshold: 0.6662827


In [22]:
# Predict using best threshold
best_preds = (
    probs >= best_threshold
).astype(int)

# Show report
print(
    classification_report(
        y_test,
        best_preds
    )
)

              precision    recall  f1-score   support

           0       0.95      0.90      0.92     56538
           1       0.27      0.44      0.33      4965

    accuracy                           0.86     61503
   macro avg       0.61      0.67      0.63     61503
weighted avg       0.89      0.86      0.87     61503



In [23]:
# Predict probabilities
probs = final_model.predict_proba(X_test)[:, 1]

# Default threshold
preds = (probs >= 0.5).astype(int)

In [24]:
# Predict probabilities
probs = final_model.predict_proba(X_test)[:, 1]

# Convert probabilities into classes
preds = (probs >= 0.5).astype(int)

# Classification report
print(classification_report(y_test, preds))

# ROC AUC score
auc = roc_auc_score(y_test, probs)

print("ROC AUC:", auc)
# Classification report
print(classification_report(y_test, preds))

# ROC AUC
auc = roc_auc_score(y_test, probs)

print("Final ROC AUC:", auc)

              precision    recall  f1-score   support

           0       0.96      0.72      0.83     56538
           1       0.18      0.70      0.29      4965

    accuracy                           0.72     61503
   macro avg       0.57      0.71      0.56     61503
weighted avg       0.90      0.72      0.78     61503

ROC AUC: 0.7814832537657834
              precision    recall  f1-score   support

           0       0.96      0.72      0.83     56538
           1       0.18      0.70      0.29      4965

    accuracy                           0.72     61503
   macro avg       0.57      0.71      0.56     61503
weighted avg       0.90      0.72      0.78     61503

Final ROC AUC: 0.7814832537657834


In [25]:
# Import precision recall
from sklearn.metrics import precision_recall_curve

# Calculate precision and recall
precision, recall, thresholds = precision_recall_curve(
    y_test,
    probs
)

# Calculate F1 scores
f1_scores = 2 * (precision * recall) / (precision + recall)

# Best threshold
best_threshold = thresholds[f1_scores[:-1].argmax()]

print("Best threshold:", best_threshold)

Best threshold: 0.6662827


In [26]:
# Model saving library
import joblib

In [27]:
# Create models directory
import os

os.makedirs(
    "models",
    exist_ok=True
)

In [28]:
# Save trained model
joblib.dump(
    final_model,
    "models/final_xgboost_credit_model.pkl"
)

['models/final_xgboost_credit_model.pkl']

In [29]:
# Save selected feature names
joblib.dump(
    selected_features.tolist(),
    "models/selected_features.pkl"
)

['models/selected_features.pkl']

In [30]:
# Save optimal threshold
joblib.dump(
    best_threshold,
    "models/best_threshold.pkl"
)

['models/best_threshold.pkl']

In [31]:
# Show saved files
os.listdir("models")

['selected_features.pkl',
 'best_threshold.pkl',
 'final_xgboost_credit_model.pkl']

In [32]:
import os

print(os.getcwd())
print(os.listdir("models"))

/kaggle/working
['selected_features.pkl', 'best_threshold.pkl', 'final_xgboost_credit_model.pkl']


In [33]:
import shutil

shutil.make_archive(
    "models",
    "zip",
    "models"
)

'/kaggle/working/models.zip'

In [34]:
os.listdir()

['__notebook__.ipynb', 'models', 'models.zip']